# Retail Data Quality — Automated Validation

## Objective

Create a reusable validation framework for the cleaned retail sales dataset.

The validation framework will test:

- Completeness
- Uniqueness
- Validity
- Referential integrity
- Financial consistency
- Date quality
- Business rules

Each test will return:

- Records tested
- Failed records
- Failure percentage
- Status
- Business interpretation

In [4]:
import pandas as pd
import numpy as np

from pathlib import Path

In [5]:
PROJECT_ROOT = Path.cwd().parent

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOCS_DIR = PROJECT_ROOT / "docs"

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [6]:
sales = pd.read_csv(
    CLEANED_DIR / "fact_sales_cleaned.csv",
    low_memory=False
)

products = pd.read_csv(
    RAW_DIR / "dim_product.csv"
)

customers = pd.read_csv(
    RAW_DIR / "dim_customer.csv"
)

stores = pd.read_csv(
    RAW_DIR / "dim_store.csv"
)

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)

Sales: (118236, 22)
Products: (500, 7)
Customers: (5000, 6)
Stores: (20, 5)


In [7]:
sales = pd.read_csv(
    CLEANED_DIR / "fact_sales_cleaned.csv",
    low_memory=False
)

products = pd.read_csv(
    RAW_DIR / "dim_product.csv"
)

customers = pd.read_csv(
    RAW_DIR / "dim_customer.csv"
)

stores = pd.read_csv(
    RAW_DIR / "dim_store.csv"
)

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)

Sales: (118236, 22)
Products: (500, 7)
Customers: (5000, 6)
Stores: (20, 5)


In [8]:
sales["transaction_date"] = pd.to_datetime(
    sales["transaction_date"],
    errors="coerce"
)

sales["quantity"] = pd.to_numeric(
    sales["quantity"],
    errors="coerce"
)

sales["unit_price"] = pd.to_numeric(
    sales["unit_price"],
    errors="coerce"
)

sales["discount_amount"] = pd.to_numeric(
    sales["discount_amount"],
    errors="coerce"
)

sales["gross_sales"] = pd.to_numeric(
    sales["gross_sales"],
    errors="coerce"
)

sales["net_sales"] = pd.to_numeric(
    sales["net_sales"],
    errors="coerce"
)

sales["unit_cost"] = pd.to_numeric(
    sales["unit_cost"],
    errors="coerce"
)

sales["cost_amount"] = pd.to_numeric(
    sales["cost_amount"],
    errors="coerce"
)

sales["profit_amount"] = pd.to_numeric(
    sales["profit_amount"],
    errors="coerce"
)

sales["gross_margin_pct"] = pd.to_numeric(
    sales["gross_margin_pct"],
    errors="coerce"
)

In [9]:
validation_results = []

In [10]:
def add_validation_result(
    test_name,
    records_tested,
    failed_records,
    status,
    business_rule
):
    failure_pct = (
        failed_records / records_tested * 100
        if records_tested > 0
        else 0
    )

    validation_results.append({
        "test": test_name,
        "records_tested": records_tested,
        "failed_records": failed_records,
        "failure_pct": round(failure_pct, 2),
        "status": status,
        "business_rule": business_rule
    })

### Validation 1 - Transaction ID not null

In [11]:
failed_mask = sales["transaction_id"].isna()

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Transaction ID not null",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule="transaction_id must not be NULL"
)

### Validation 2 - Transaction ID unique

In [12]:
failed_records = sales["transaction_id"].duplicated().sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Transaction ID unique",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule="transaction_id must be unique"
)

### Validation 3 - Quantity positive

In [13]:
failed_mask = (
    sales["quantity"].isna()
    |
    (sales["quantity"] <= 0)
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Quantity greater than zero",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule="quantity must be greater than 0"
)

### Validation 4 - Unit price positive

In [14]:
failed_mask = (
    sales["unit_price"].isna()
    |
    (sales["unit_price"] <= 0)
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Unit price valid",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule="unit_price must be greater than 0"
)

### Validation 5 - Discount valid

In [15]:
failed_mask = (
    sales["discount_amount"].isna()
    |
    (sales["discount_amount"] < 0)
    |
    (
        sales["discount_amount"]
        > sales["gross_sales"]
    )
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Discount valid",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "discount_amount must be >= 0 and "
        "must not exceed gross_sales"
    )
)

### Validation 6 - Transaction date valid

In [16]:
MIN_DATE = pd.Timestamp("2024-01-01")
MAX_DATE = pd.Timestamp("2025-12-31")

In [17]:
failed_mask = (
    sales["transaction_date"].isna()
    |
    (sales["transaction_date"] < MIN_DATE)
    |
    (sales["transaction_date"] > MAX_DATE)
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Transaction date valid",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "transaction_date must exist and be between "
        "2024-01-01 and 2025-12-31"
    )
)

### Validation 7 - Product referential integrity

In [18]:
valid_product_ids = set(
    products["product_id"]
)

failed_mask = (
    ~sales["product_id"].isin(
        valid_product_ids
    )
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Product referential integrity",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "Every product_id must exist in dim_product"
    )
)

In [19]:
failed_mask = (
    (sales["product_id"] != "UNKNOWN")
    &
    ~sales["product_id"].isin(valid_product_ids)
)

failed_records = failed_mask.sum()

In [20]:
status = "PASS" if failed_records == 0 else "FAIL"

### Validation 8 - Customer referential integrity

In [21]:
valid_customer_ids = set(
    customers["customer_id"]
)

failed_mask = (
    (sales["customer_id"] != "UNKNOWN")
    &
    ~sales["customer_id"].isin(
        valid_customer_ids
    )
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Customer referential integrity",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "Every customer_id other than UNKNOWN "
        "must exist in dim_customer"
    )
)

### Validation 9 - Store referential integrity

In [22]:
valid_store_ids = set(
    stores["store_id"]
)

failed_mask = (
    (sales["store_id"] != "UNKNOWN")
    &
    ~sales["store_id"].isin(valid_store_ids)
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Store referential integrity",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "Every store_id other than UNKNOWN "
        "must exist in dim_store"
    )
)

### Validation 10 - Gross sales calculation

In [23]:
expected_gross_sales = (
    sales["quantity"]
    * sales["unit_price"]
).round(2)

In [24]:
failed_mask = ~np.isclose(
    sales["gross_sales"],
    expected_gross_sales,
    rtol=0,
    atol=0.01
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Gross sales calculation",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "gross_sales = quantity × unit_price"
    )
)

### Validation 11 - Net sales calculation

In [25]:
expected_net_sales = (
    sales["gross_sales"]
    - sales["discount_amount"]
).round(2)

failed_mask = ~np.isclose(
    sales["net_sales"],
    expected_net_sales,
    rtol=0,
    atol=0.01
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Net sales calculation",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "net_sales = gross_sales - discount_amount"
    )
)

### Validation 12 - Cost calculation

In [26]:
expected_cost = (
    sales["quantity"]
    * sales["unit_cost"]
).round(2)

failed_mask = ~np.isclose(
    sales["cost_amount"],
    expected_cost,
    rtol=0,
    atol=0.01
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Cost calculation",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "cost_amount = quantity × unit_cost"
    )
)

### Validation 13 - Profit calculation

In [27]:
expected_profit = (
    sales["net_sales"]
    - sales["cost_amount"]
).round(2)

failed_mask = ~np.isclose(
    sales["profit_amount"],
    expected_profit,
    rtol=0,
    atol=0.01
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Profit calculation",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "profit_amount = net_sales - cost_amount"
    )
)

### Validation 14 - Gross margin calculation

In [28]:
expected_margin = np.where(
    sales["net_sales"] != 0,
    sales["profit_amount"]
    / sales["net_sales"]
    * 100,
    np.nan
)

failed_mask = ~np.isclose(
    sales["gross_margin_pct"],
    expected_margin,
    rtol=0,
    atol=0.01,
    equal_nan=True
)

failed_records = failed_mask.sum()

status = "PASS" if failed_records == 0 else "FAIL"

add_validation_result(
    test_name="Gross margin calculation",
    records_tested=len(sales),
    failed_records=failed_records,
    status=status,
    business_rule=(
        "gross_margin_pct = profit / net_sales × 100"
    )
)

### Validation 15 - Unknown attribute review

In [29]:
unknown_customer = (
    sales["customer_id"] == "UNKNOWN"
).sum()

unknown_product = (
    sales["product_id"] == "UNKNOWN"
).sum()

unknown_store = (
    sales["store_id"] == "UNKNOWN"
).sum()

In [30]:
unknown_attribution_count = (
    unknown_customer
    + unknown_product
    + unknown_store
)

In [31]:
validation_results.append({
    "test": "Unknown reference attribution",
    "records_tested": len(sales),
    "failed_records": unknown_attribution_count,
    "failure_pct": round(
        unknown_attribution_count
        / len(sales)
        * 100,
        2
    ),
    "status": (
        "REVIEW"
        if unknown_attribution_count > 0
        else "PASS"
    ),
    "business_rule": (
        "UNKNOWN reference values should be monitored "
        "and investigated with the source system."
    )
})

In [32]:
validation_report = pd.DataFrame(
    validation_results
)

In [33]:
validation_results = pd.DataFrame(
    validation_results
)

In [39]:
validation_results

,test,records_tested,failed_records,failure_pct,status,business_rule
0,Transaction ID not null,118236,0,0.00,PASS,transaction_id must not be NULL
1,Transaction ID unique,118236,0,0.00,PASS,transaction_id must be unique
2,Quantity greater than zero,118236,0,0.00,PASS,quantity must be greater than 0
3,Unit price valid,118236,0,0.00,PASS,unit_price must be greater than 0
4,Discount valid,118236,0,0.00,PASS,discount_amount must be >= 0 and must not exce...
5,Transaction date valid,118236,0,0.00,PASS,transaction_date must exist and be between 202...
6,Product referential integrity,118236,147,0.12,FAIL,Every product_id must exist in dim_product
7,Customer referential integrity,118236,0,0.00,PASS,Every customer_id other than UNKNOWN must exis...
8,Store referential integrity,118236,0,0.00,PASS,Every store_id other than UNKNOWN must exist i...
9,Gross sales calculation,118236,0,0.00,PASS,gross_sales = quantity × unit_price


In [40]:
validation_results.to_csv(
    "../data/processed/data_validation_results.csv",
    index=False
)

print("Validation results saved successfully.")

Validation results saved successfully.


In [34]:
validation_report

,test,records_tested,failed_records,failure_pct,status,business_rule
0,Transaction ID not null,118236,0,0.00,PASS,transaction_id must not be NULL
1,Transaction ID unique,118236,0,0.00,PASS,transaction_id must be unique
2,Quantity greater than zero,118236,0,0.00,PASS,quantity must be greater than 0
3,Unit price valid,118236,0,0.00,PASS,unit_price must be greater than 0
4,Discount valid,118236,0,0.00,PASS,discount_amount must be >= 0 and must not exce...
5,Transaction date valid,118236,0,0.00,PASS,transaction_date must exist and be between 202...
6,Product referential integrity,118236,147,0.12,FAIL,Every product_id must exist in dim_product
7,Customer referential integrity,118236,0,0.00,PASS,Every customer_id other than UNKNOWN must exis...
8,Store referential integrity,118236,0,0.00,PASS,Every store_id other than UNKNOWN must exist i...
9,Gross sales calculation,118236,0,0.00,PASS,gross_sales = quantity × unit_price


In [35]:
validation_report["status"].value_counts()

status
PASS      13
FAIL       1
REVIEW     1
Name: count, dtype: int64

In [36]:
total_tests = len(validation_report)

passed_tests = (
    validation_report["status"] == "PASS"
).sum()

failed_tests = (
    validation_report["status"] == "FAIL"
).sum()

review_tests = (
    validation_report["status"] == "REVIEW"
).sum()

print(f"Total tests: {total_tests}")
print(f"PASS: {passed_tests}")
print(f"FAIL: {failed_tests}")
print(f"REVIEW: {review_tests}")

Total tests: 15
PASS: 13
FAIL: 1
REVIEW: 1


In [37]:
data_quality_score = (
    passed_tests
    / total_tests
    * 100
)

print(
    f"Validation test pass rate: "
    f"{data_quality_score:.2f}%"
)

Validation test pass rate: 86.67%


In [38]:
validation_path = (
    DOCS_DIR / "validation_report.csv"
)

validation_report.to_csv(
    validation_path,
    index=False
)

print(
    f"Validation report saved: {validation_path}"
)

Validation report saved: d:\GitHub\retail_data_quality_project\docs\validation_report.csv
